# Retrieval and answering examples
This notebook reads saved artifacts only. It performs no new retrieval or generation.

## Load artifacts

In [1]:
import json
import os
from pathlib import Path

from potency_kg.notebook_present import validate_saved_citations

package_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
results_root = Path(os.environ.get("POTENCY_KG_RESULTS_ROOT", "results")).expanduser()
if not results_root.is_absolute():
    results_root = package_root / results_root
paths = {
    "answers": results_root / "answers.jsonl",
    "traces": results_root / "traces.jsonl",
    "fixture_report": results_root / "fixtures" / "report.json",
}
print("Results root:", results_root)
print("Available run files:")
for label, path in paths.items():
    print(f"  {label}: {path} ({'present' if path.is_file() else 'missing'})")

def read_jsonl(path):
    if not path.is_file():
        return []
    try:
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    except (OSError, UnicodeError, json.JSONDecodeError) as error:
        print(f"Could not read saved artifact {path}: {type(error).__name__}")
        return []

def read_json(path):
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, UnicodeError, json.JSONDecodeError) as error:
        print(f"Could not read saved artifact {path}: {type(error).__name__}")
        return {}

answers = read_jsonl(paths["answers"])
traces = read_jsonl(paths["traces"])
if not answers or not traces:
    print("Missing saved artifacts: run answer and fixture answer stages first, then reopen this notebook.")

Results root: /workspace/results
Available run files:
  answers: /workspace/results/answers.jsonl (present)
  traces: /workspace/results/traces.jsonl (present)
  fixture_report: /workspace/results/fixtures/report.json (present)


## P3 retrieval comparison

In [2]:
p3_answers = {row.get("mode"): row for row in answers if row.get("question_id") == "P3"}
p3_traces = {row.get("mode"): row for row in traces if row.get("question_id") == "P3"}
if not all(mode in p3_answers and mode in p3_traces for mode in ("vector", "graph")):
    print("Missing saved artifacts: P3 vector and graph answers/traces are required for comparison.")
else:
    print("P3 vector versus graph")
    for mode in ("vector", "graph"):
        answer = p3_answers[mode]
        trace = p3_traces[mode]
        evidence = trace.get("evidence_text", "")
        print(f"mode={mode} graph_version={trace.get('graph_version')} index_version={trace.get('index_version')}")
        print(f"  selected_passage_ids={trace.get('selected_passage_ids', [])}")
        print(f"  selected_via={trace.get('selected_via', [])}")
        print(f"  chars_used={len(evidence)} max_evidence_chars={trace.get('max_evidence_chars')}")
        print(f"  citations={answer.get('citations', [])}")
        for block in evidence.split("\n\n"):
            excerpt = block.partition("\n")[2][:120]
            if excerpt:
                print(f"  saved evidence excerpt: {excerpt}")

P3 vector versus graph
mode=vector graph_version=graph-307b3797ee0fe8db7151aabddc53c8616003d64a8e6b4f6cad79da6331d98c41 index_version=index-563432d174c8967fa01b316ce20e425d93b249e5a20ec52f68ce3b4cda129f45
  selected_passage_ids=['sbra-casgevy-tdt-2024-01:page-0018-c02', 'sbra-casgevy-scd-2023-12:page-0005', 'sbra-casgevy-scd-2023-12:page-0018']
  selected_via=[{'passage_id': 'sbra-casgevy-tdt-2024-01:page-0018-c02', 'selected_via': 'vector', 'source_id': 'sbra-casgevy-tdt-2024-01', 'source_version': 'sha256:1332bc62dc6eb3a9e8e960a53cc9225fccac968d76d0e8269d942254e743982f'}, {'passage_id': 'sbra-casgevy-scd-2023-12:page-0005', 'selected_via': 'vector', 'source_id': 'sbra-casgevy-scd-2023-12', 'source_version': '2023-12-08:0f9bf94c6edd4474'}, {'passage_id': 'sbra-casgevy-scd-2023-12:page-0018', 'selected_via': 'vector', 'source_id': 'sbra-casgevy-scd-2023-12', 'source_version': '2023-12-08:0f9bf94c6edd4474'}]
  chars_used=6000 max_evidence_chars=6000
  citations=[{'match': 'normalized', 

## Citation example

In [3]:
if not p3_answers or not p3_traces:
    print("Missing saved artifacts: no P3 citation example is available.")
else:
    print("P3 citation-tag and quote validation against saved evidence_text")
    for mode in ("vector", "graph"):
        if mode not in p3_answers or mode not in p3_traces:
            continue
        check, citations = validate_saved_citations(p3_answers[mode], p3_traces[mode])
        print(f"mode={mode} citation_check={check['status']} failures={check['failures']}")
        for cited in citations:
            print(f"  tag={cited['tag']} quote={cited['quote']!r} match={cited['match']} normalized_match={cited['normalized_match']}")

P3 citation-tag and quote validation against saved evidence_text
mode=vector citation_check=passed failures=[]
  tag=S1 quote='Several off-target loci contributed by variants were not empirically tested by the Applicant.' match=normalized normalized_match=True
  tag=S1 quote='the review team required a post-marketing bioinformatics study for CASGEVY to adequately assess potential off-target editing risks' match=normalized normalized_match=True
  tag=S1 quote='two postmarketing requirements (PMRs) under 505(o) of the Federal Food, Drug, and Cosmetic Act' match=normalized normalized_match=True
  tag=S1 quote='Long Term Follow-up After Administration of Human Gene Therapy Products (January 2020)' match=normalized normalized_match=True
  tag=S2 quote='CASGEVY is a biological product containing genetically modified autologous HSPCs that' match=exact normalized_match=False
  tag=S3 quote='The Applicant will conduct routine pharmacovigilance in accordance with 21 CFR 600.80' match=normalized 

## Fixture-history example

In [4]:
history = {}
for stage in ("before", "after"):
    base = results_root / "fixtures" / "history" / "source-correction" / stage
    manifest = read_json(base / "manifest.json")
    stage_answers = read_jsonl(base / "answers.jsonl")
    stage_traces = read_jsonl(base / "traces.jsonl")
    retrieval = read_jsonl(base / "retrieval-traces.jsonl")
    if not manifest or not stage_answers or not stage_traces or not retrieval:
        print(f"Missing saved artifacts: correction {stage} history is incomplete.")
        continue
    history[stage] = {"manifest": manifest, "answer": stage_answers[0],
                      "trace": stage_traces[0], "retrieval": retrieval[0]}
    citations = stage_answers[0].get("citations", [])
    print(f"{stage}: source_versions={manifest.get('source_versions', [])}")
    print(f"  selected_passage_ids={stage_traces[0].get('selected_passage_ids', [])}")
    print(f"  source-version citations={[(item.get('source_id'), item.get('source_version'), item.get('passage_id')) for item in citations]}")
if "before" in history and "after" in history:
    before_v1 = {(item.get("source_id"), item.get("source_version"), item.get("passage_id"))
                 for item in history["before"]["retrieval"].get("passages", [])
                 if item.get("source_version") == "fixture-v1"}
    after_passages = {(item.get("source_id"), item.get("source_version"), item.get("passage_id"))
                      for item in history["after"]["retrieval"].get("passages", [])}
    print(f"superseded v1 passage absent after replacement: {bool(before_v1) and before_v1.isdisjoint(after_passages)}")
conflict_base = results_root / "fixtures" / "history" / "unresolved-conflict" / "combined"
if (conflict_base / "manifest.json").is_file():
    conflict_answers = read_jsonl(conflict_base / "answers.jsonl")
    report = read_json(paths["fixture_report"])
    if conflict_answers:
        print("conflict cited source versions:",
              [(item.get("source_id"), item.get("source_version"))
               for item in conflict_answers[0].get("citations", [])])
        print("recorded unresolved-conflict check:",
              report.get("checks", {}).get("conflict_unresolved_wording_heuristic", "not available"))

before: source_versions=[{'source_id': 'assay-attribute-note', 'source_version': 'fixture-v1'}]
  selected_passage_ids=['assay-attribute-note:p0001']
  source-version citations=[('assay-attribute-note', 'fixture-v1', 'assay-attribute-note:p0001'), ('assay-attribute-note', 'fixture-v1', 'assay-attribute-note:p0001'), ('assay-attribute-note', 'fixture-v1', 'assay-attribute-note:p0001'), ('assay-attribute-note', 'fixture-v1', 'assay-attribute-note:p0001')]
after: source_versions=[{'source_id': 'assay-attribute-note', 'source_version': 'fixture-v2'}]
  selected_passage_ids=['assay-attribute-note:p0001']
  source-version citations=[('assay-attribute-note', 'fixture-v2', 'assay-attribute-note:p0001'), ('assay-attribute-note', 'fixture-v2', 'assay-attribute-note:p0001'), ('assay-attribute-note', 'fixture-v2', 'assay-attribute-note:p0001')]
superseded v1 passage absent after replacement: True
conflict cited source versions: [('plan-k-note-b', 'fixture-v1'), ('plan-k-note-a', 'fixture-v1'), ('p

## Reproducibility note

In [5]:
if p3_traces:
    trace = p3_traces.get("graph") or next(iter(p3_traces.values()))
    print("config_hash:", trace.get("config_hash"))
    print("prompt_sha256:", trace.get("prompt_sha256"))
    print("prompt_version:", trace.get("prompt_version"))
    print("retriever_version:", trace.get("retriever_version"))
    print("corpus_snapshot:", trace.get("corpus_snapshot"))
    print("saved artifact paths:", paths["answers"], paths["traces"],
          results_root / "fixtures" / "history" / "source-correction")
print("This notebook performs no new retrieval or generation.")

config_hash: 4ab6cf693c09b656623c2c024deddb016dfe44c870d0617f3e01eca54c06b029
prompt_sha256: 3c10d8aee0345061c2de0eef84be1d7cfc18298c3bcb20b252485285aacd7586
prompt_version: answer-prompt-v1
retriever_version: similarity-pruned-beam-v1
corpus_snapshot: potency-assignment-2026-09-21-v3
saved artifact paths: /workspace/results/answers.jsonl /workspace/results/traces.jsonl /workspace/results/fixtures/history/source-correction
This notebook performs no new retrieval or generation.
